In [ ]:
# Imports
import torch
import pickle
import glob
import pandas as pd
import numpy as np
import os
import sys
import subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor


In [ ]:
# Create directories
!mkdir -p results work_dir models data

In [ ]:
# Empty folders
!rm -rf /kaggle/working/data/*
!rm -rf /kaggle/working/models/*

In [ ]:
# Check devices for training
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Copies code from github (https://github.com/moeschl21/TransformerLSR_moeschl.git) into the working directory 
!git clone https://github.com/moeschl21/TransformerLSR_moeschl.git
!cp -r TransformerLSR_moeschl/* /kaggle/working/
!rm -rf TransformerLSR_moeschl

In [ ]:
# Setting up variables bei 59 STARTEN!!
num_sim = 100 # Number of Simulations
num_traj = 100 # Number of patients
timeout = 1000 # Maximum visits
batch_size = 16
num_enc_layer = 2 #2
num_dec_layer = 3 #3
lr = 0.0001 #0.0001
model_size = 64 # Embedding size 64
epoch = 100
#ffn = 64

In [ ]:
!ls /kaggle/input/datasets/jonasmschl/data-lsr-59/data_lsr/

In [ ]:
!cp -r /kaggle/input/datasets/jonasmschl/data-lsr-59/data_lsr/. /kaggle/working/data/

In [ ]:
########################################################################
############################### TRAINING ###############################
########################################################################

In [ ]:
import logging

logger = logging.getLogger()

for handler in logger.handlers[:]:
    logger.removeHandler(handler)

In [ ]:
# Training with missing Y_1 25%, Y_2 15% and Y3 3%

# Simple Training
for seed in range(1,5 + 1):
    get_ipython().run_line_magic(
        "run",
        f"main.py "
        f"--seed {seed} "
        f"--epoch {epoch} "
        f"--batch_size {batch_size} "
        f"--num_enc_layer {num_enc_layer} "
        f"--num_dec_layer {num_dec_layer} "
        f"--lr {lr} "
        f"--model_size {model_size} "
        f"--local "
        f"--data DIVAT_sim_missing_{num_traj}_visit_{timeout}_long_3"
    )

    if seed == num_sim:
        print("done")
    
    logger = logging.getLogger()

    for handler in logger.handlers[:]:
        logger.removeHandler(handler)



In [ ]:
# For Parallel training
seeds = list(range(96, 101))

# Check GPUs
gpu_count = torch.cuda.device_count()
print(f"Available GPUs: {gpu_count}")

for i in range(gpu_count):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

if gpu_count < 2:
    raise RuntimeError(
        "2 GPUs needed"
    )

# Load data
data_name = (
    f"DIVAT_sim_missing_{num_traj}_visit_"
    f"{timeout}_long_3"
)

os.makedirs("training_logs", exist_ok=True)

# Functions for training
def train_seed(seed, gpu_id):

    # each GPU
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    env["PYTHONUNBUFFERED"] = "1"

    command = [
        sys.executable,
        "-u",
        "main.py",
        "--seed", str(seed),
        "--epoch", str(epoch),
        "--batch_size", str(batch_size),
        "--num_enc_layer", str(num_enc_layer),
        "--num_dec_layer", str(num_dec_layer),
        "--lr", str(lr),
        "--model_size", str(model_size),
        "--local",
        "--data", data_name
    ]

    log_path = f"training_logs/seed_{seed}_gpu_{gpu_id}.log"

    print(
        f"\n[GPU {gpu_id} | Seed {seed}] "
        f"Training started",
        flush=True
    )

    # Start training
    with open(log_path, "w", buffering=1) as log_file:

        process = subprocess.Popen(
            command,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        for line in process.stdout:
            line = line.rstrip("\n")

            # logs
            log_file.write(line + "\n")

            # logs in kaggle
            print(
                f"[GPU {gpu_id} | Seed {seed}] {line}",
                flush=True
            )

        returncode = process.wait()

    if returncode != 0:
        print(
            f"[GPU {gpu_id} | Seed {seed}] Failed!",
            flush=True
        )
        raise RuntimeError(
            f"Seed {seed} failed. "
            f"See {log_path}"
        )

    print(
        f"\n[GPU {gpu_id} | Seed {seed}] "
        f"Training done!",
        flush=True
    )

# Function for choosen GPU
def train_gpu(gpu_id, assigned_seeds):

    successful = []
    failed = []

    for seed in assigned_seeds:

        try:
            train_seed(seed, gpu_id)
            successful.append(seed)

        except Exception as e:
            failed.append(seed)
            print(
                f"\n[GPU {gpu_id} | Seed {seed}] "
                f"Failed: {e}",
                flush=True
            )

    return successful, failed


print("\n" + "=" * 60)
print("PARALLEL TRANSFORMERLSR TRAINING")
print("=" * 60)

print(f"Seeds: {seeds}")
print(f"Epochs: {epoch}")
print(f"Batch Size: {batch_size}")
print(f"Dataset: {data_name}")
print("GPUs: 2")
print("=" * 60 + "\n")

# Seeds divided between GPUs
gpu0_seeds = seeds[::2]
gpu1_seeds = seeds[1::2]

print(f"GPU 0 trains Seeds: {gpu0_seeds}")
print(f"GPU 1 trains Seeds: {gpu1_seeds}")
print()

with ThreadPoolExecutor(max_workers=2) as executor:

    gpu0 = executor.submit(
        train_gpu, 0, gpu0_seeds
    )

    gpu1 = executor.submit(
        train_gpu, 1, gpu1_seeds
    )

    success0, failed0 = gpu0.result()
    success1, failed1 = gpu1.result()


# Results overview
successful_seeds = sorted(success0 + success1)
failed_seeds = sorted(failed0 + failed1)

print("\n" + "=" * 60)
print("TRAINING DONE")
print("=" * 60)

print(f"Seeds: {successful_seeds}")
print(f"Failed Seeds: {failed_seeds}")

if len(failed_seeds) == 0:
    print("\n All others successful!")
else:
    print("Failed seeds!")

print("=" * 60)
